Training pipeline for FUTVE match outcome prediction.

Trains and evaluates multiple classifiers (CatBoost, Random Forest,
Logistic Regression) using a chronological train/test split to
prevent data leakage

In [2]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

In [3]:
DATA_DIR = Path().resolve().parent / "data"
FEATURES_CSV = DATA_DIR / "futve_features.csv"
MODELS_DIR = DATA_DIR / "models"

FEATURE_COLUMNS = [
    "home_last5_wins",
    "home_last5_draws",
    "home_last5_losses",
    "away_last5_wins",
    "away_last5_draws",
    "away_last5_losses",
    "home_avg_goals",
    "home_avg_goals_conceded",
    "away_avg_goals",
    "away_avg_goals_conceded",
    "home_avg_goals_last5",
    "home_avg_conceded_last5",
    "away_avg_goals_last5",
    "away_avg_conceded_last5",
    "home_form_points",
    "away_form_points",
    "home_elo",
    "away_elo",
    "elo_diff",
    "home_rank",
    "away_rank",
    "home_win_streak",
    "home_draw_streak",
    "home_loss_streak",
    "home_unbeaten_streak",
    "away_win_streak",
    "away_draw_streak",
    "away_loss_streak",
    "away_unbeaten_streak",
    "h2h_home_wins",
    "h2h_away_wins",
    "h2h_draws",
    "h2h_total",
    "home_home_win_rate",
    "home_home_avg_goals",
    "away_away_win_rate",
    "away_away_avg_goals",
    "home_rest_days",
    "away_rest_days",
    "home_matches_played",
    "away_matches_played",
    "home_goal_diff",
    "away_goal_diff",
]

TARGET = "result"
LABELS = ["H", "D", "A"]

# Minimum matches a team must have played before we use that row for training,
# so the features are not all zeros.
MIN_MATCHES_PLAYED = 5

# Chronological split: use last N seasons as test set.
TEST_SEASONS = 4

In [ ]:
def load_features() -> pd.DataFrame:

    df = pd.read_csv(FEATURES_CSV)
    df["match_date_utc"] = pd.to_datetime(df["match_date_utc"], utc=True)
    return df

def filter_cold_start(df: pd.DataFrame) -> pd.DataFrame:

    mask = (df["home_matches_played"] >= MIN_MATCHES_PLAYED) & (df["away_matches_played"] >= MIN_MATCHES_PLAYED)

    filtered = df[mask].reset_index(drop=True)


    return filtered

def temporal_split(df: pd.DataFrame):

    seasons = sorted(df["season"].unique())
    test_seasons = set(seasons[-TEST_SEASONS:])
    train_seasons = set(seasons) - test_seasons

    


In [ ]:
print("=" * 60)
print("  FUTVE Match Prediction - Model Training")
print("=" * 60)

print(f"\nLoading features from {FEATURES_CSV}")
df = load_features()
print(f"  -> {len(df)} rows, {len(df.columns)} columns")

print("\nFiltering cold-start matches...")
df = filter_cold_start(df)

print("\nSplitting data (temporal split)...")
train_df, test_df = temporal_split(df)


  FUTVE Match Prediction - Model Training

Loading features from C:\Users\Carlos\OneDrive\Documentos\MLFutvePredictions\data\futve_features.csv
  -> 6592 rows, 49 columns

Filtering cold-start matches...
